# Prédiction Multi-Source avec `AttentionDeepSetsDiagnostic` (`../models/detection_error_src.pt`)

Ce notebook est dédié uniquement à **la prédiction / inférence en production** à partir du checkpoint unique `../models/detection_error_src.pt` :
- **Mode 1 :** Prédiction sur un **nouveau fichier CSV** ou un sous-ensemble de données (regroupé par `Batch_ID`, `Cable_ID`, `Operation_Type` ou toute autre colonne).
- **Mode 2 :** Prédiction directe sur une **séquence d'événements manuelle** (liste de dictionnaires Python représentant les étapes d'un câble ou d'un lot) **sans avoir besoin de colonnes cibles**.
- **Export :** Sauvegarde des prédictions (probabilités + entités détectées au-dessus du seuil $C$) dans un fichier CSV.

In [6]:
import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Appareil utilisé :', device)

Appareil utilisé : cpu


## 1. Chargement du modèle (`../models/detection_error_src.pt`) et fonction de prédiction

In [7]:
class AttentionDeepSetsDiagnostic(nn.Module):
    def __init__(self, in_features: int, hidden_dim: int = 128, num_targets: int = 21):
        super().__init__()
        self.phi = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU()
        )
        self.attn_score = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, num_targets)
        )
        self.rho = nn.Sequential(
            nn.Linear(hidden_dim + hidden_dim * num_targets, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(hidden_dim * 2, num_targets)
        )
        self.num_targets = num_targets

    def forward(self, x_events: torch.Tensor, group_idx: torch.Tensor, num_groups: int) -> torch.Tensor:
        h = self.phi(x_events)
        dev = x_events.device
        dim = h.size(1)

        # 1. Mean Pooling global
        sum_pool = torch.zeros((num_groups, dim), device=dev, dtype=h.dtype)
        sum_pool.index_add_(0, group_idx, h)
        counts = torch.zeros((num_groups, 1), device=dev, dtype=h.dtype)
        counts.index_add_(0, group_idx, torch.ones((h.size(0), 1), device=dev, dtype=h.dtype))
        mean_pool = sum_pool / counts.clamp(min=1.0)

        # 2. Attention Pooling par cible
        scores = self.attn_score(h)
        max_scores = torch.full((num_groups, self.num_targets), -1e9, device=dev, dtype=h.dtype)
        max_scores.scatter_reduce_(
            0, group_idx.unsqueeze(1).expand(-1, self.num_targets), scores, reduce='amax', include_self=True
        )
        exp_scores = torch.exp(scores - max_scores[group_idx])
        denom = torch.zeros((num_groups, self.num_targets), device=dev, dtype=h.dtype)
        denom.index_add_(0, group_idx, exp_scores)
        attn_weights = exp_scores / denom[group_idx].clamp(min=1e-8)

        weighted_h = attn_weights.unsqueeze(-1) * h.unsqueeze(1)
        attn_pool = torch.zeros((num_groups, self.num_targets, dim), device=dev, dtype=h.dtype)
        attn_pool.index_add_(0, group_idx, weighted_h)
        attn_flat = attn_pool.reshape(num_groups, -1)

        # 3. Fusion et prédiction
        combined = torch.cat([mean_pool, attn_flat], dim=1)
        return self.rho(combined)

CHECKPOINT_PATH = '../models/detection_error_src.pt'
if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(f'Checkpoint introuvable : {CHECKPOINT_PATH}')

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
model = AttentionDeepSetsDiagnostic(
    in_features=checkpoint['in_features'],
    hidden_dim=checkpoint['hidden_dim'],
    num_targets=checkpoint['num_targets']
).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

scaler = checkpoint['scaler']
FEATURE_COLS = checkpoint['feature_cols']
ALL_ENTITIES = checkpoint['entities']
DEFAULT_THRESHOLD = 0.60  # Compromis optimal observé (0.50 à 0.80)

NUM_COLS = [
    'Step_Order', 'Total_Cable_Steps', 'Num_Machine_Ops', 'Num_Manual_Ops',
    'Hours_Into_Shift',
    'Machine_Hours_Since_Maint', 'Machine_Vibration_mm_s', 'Machine_Temp_C', 'Machine_Cycle_Time_sec',
    'Operator_Exp_Months', 'Operator_Handling_Time_sec',
    'Sensor_Calibration_Drift', 'Sensor_Anomaly_Score', 'Sensor_Error_Flag',
    'Inspection_Duration_sec', 'Human_Error_Flag',
    'Step_Observed_Error', 'Confirmed_Physical_Defect'
]
CAT_COLS = ['Shift', 'Operation_Mode', 'Operation_Type', 'Machine_ID', 'Operator_ID', 'Sensor_ID', 'Inspector_ID', 'Control_Mode']

class InferenceSetDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        gid, feats = self.samples[idx]
        return gid, torch.tensor(feats, dtype=torch.float32)

def collate_inference(batch):
    gids, feat_list = zip(*batch)
    group_ids = [torch.full((f.size(0),), i, dtype=torch.long) for i, f in enumerate(feat_list)]
    return gids, torch.cat(feat_list, dim=0), torch.cat(group_ids, dim=0), len(feat_list)

def predict_error_sources(df_events: pd.DataFrame, group_col: str = 'Batch_ID', threshold: float = DEFAULT_THRESHOLD):
    """Prédit les probabilités et les entités en erreur pour chaque groupe de `group_col`."""
    df_work = df_events.copy()
    for col in NUM_COLS:
        if col not in df_work.columns:
            df_work[col] = 0.0
    for col in CAT_COLS:
        if col not in df_work.columns:
            df_work[col] = 'NONE'

    df_num = df_work[NUM_COLS].fillna(0.0)
    df_cat = pd.get_dummies(df_work[CAT_COLS].fillna('NONE'), dtype=np.float32)
    X_raw = pd.concat([df_num, df_cat], axis=1)

    for col in FEATURE_COLS:
        if col not in X_raw.columns:
            X_raw[col] = 0.0
    X_raw = X_raw[FEATURE_COLS].astype(np.float32)

    X_scaled = pd.DataFrame(scaler.transform(X_raw), index=X_raw.index, columns=FEATURE_COLS, dtype=np.float32)
    X_scaled[group_col] = df_work[group_col].values

    samples = [
        (gid, grp[FEATURE_COLS].values.astype(np.float32))
        for gid, grp in X_scaled.groupby(group_col, sort=False)
        if len(grp) > 0
    ]

    loader = DataLoader(InferenceSetDataset(samples), batch_size=32, shuffle=False, collate_fn=collate_inference)
    all_gids, all_probs = [], []
    with torch.no_grad():
        for gids, x_cat, g_idx, n_grp in loader:
            probs = torch.sigmoid(model(x_cat.to(device), g_idx.to(device), n_grp)).cpu().numpy()
            all_gids.extend(gids)
            all_probs.append(probs)

    probs_df = pd.DataFrame(np.vstack(all_probs), index=all_gids, columns=ALL_ENTITIES)
    probs_df.index.name = group_col

    # Résumé lisible des sources détectées au-dessus du seuil
    summary_rows = []
    for gid, row in probs_df.iterrows():
        detected = row[row >= threshold].sort_values(ascending=False)
        top3 = row.sort_values(ascending=False).head(3)
        summary_rows.append({
            group_col: gid,
            'Nb_Sources_Detectees': len(detected),
            'Sources_Detectees': ', '.join([f'{k} ({v:.2f})' for k, v in detected.items()]) if len(detected) > 0 else 'Aucune',
            'Top_3_Probabilites': ', '.join([f'{k}: {v:.2f}' for k, v in top3.items()])
        })
    summary_df = pd.DataFrame(summary_rows)
    return probs_df, summary_df

print(f'Modèle prêt pour la prédiction | Seuil par défaut : {DEFAULT_THRESHOLD}')

Modèle prêt pour la prédiction | Seuil par défaut : 0.6


## 2. Mode 1 : Prédiction sur un fichier CSV (par `Batch_ID`, `Cable_ID` ou `Operation_Type`)

In [8]:
INPUT_CSV = 'D:/S5/Projet_d_expertise/projet/data/processed/detection_error_src.csv'
df_input = pd.read_csv(INPUT_CSV)

# Exemple A : Prédire sur les 15 premiers lots (Batch_ID)
sample_batches = df_input['Batch_ID'].unique()[:15]
df_sub_batch = df_input[df_input['Batch_ID'].isin(sample_batches)]

probs_batch_df, summary_batch_df = predict_error_sources(df_sub_batch, group_col='Batch_ID', threshold=0.60)
print('=== Diagnostic par Batch_ID (Seuil C = 0.60) ===')
display(summary_batch_df)

# Exemple B : Prédire sur les 10 premiers câbles (Cable_ID) sans réentraîner
sample_cables = df_input['Cable_ID'].unique()[:10]
df_sub_cable = df_input[df_input['Cable_ID'].isin(sample_cables)]

probs_cable_df, summary_cable_df = predict_error_sources(df_sub_cable, group_col='Cable_ID', threshold=0.60)
print('\n=== Diagnostic par Cable_ID (Seuil C = 0.60) ===')
display(summary_cable_df)

=== Diagnostic par Batch_ID (Seuil C = 0.60) ===


,Batch_ID,Nb_Sources_Detectees,Sources_Detectees,Top_3_Probabilites
0,LOT_0001,0,Aucune,"INSP_02: 0.29, INSP_01: 0.09, INSP_03: 0.05"
1,LOT_0002,2,"MACH_02 (0.98), SENS_02 (0.92)","MACH_02: 0.98, SENS_02: 0.92, SENS_03: 0.10"
2,LOT_0003,1,MACH_03 (0.91),"MACH_03: 0.91, SENS_01: 0.20, OP_05: 0.16"
3,LOT_0004,0,Aucune,"OP_03: 0.53, OP_08: 0.27, OP_01: 0.26"
4,LOT_0005,1,SENS_03 (0.99),"SENS_03: 0.99, MACH_02: 0.08, MACH_01: 0.05"
5,LOT_0006,1,MACH_05 (1.00),"MACH_05: 1.00, OP_02: 0.51, OP_04: 0.32"
6,LOT_0007,0,Aucune,"INSP_02: 0.14, INSP_03: 0.04, MACH_05: 0.02"
7,LOT_0008,2,"OP_04 (0.88), OP_01 (0.62)","OP_04: 0.88, OP_01: 0.62, OP_08: 0.19"
8,LOT_0009,2,"MACH_03 (1.00), OP_08 (0.63)","MACH_03: 1.00, OP_08: 0.63, OP_05: 0.45"
9,LOT_0010,0,Aucune,"MACH_05: 0.35, MACH_03: 0.23, SENS_03: 0.19"



=== Diagnostic par Cable_ID (Seuil C = 0.60) ===


,Cable_ID,Nb_Sources_Detectees,Sources_Detectees,Top_3_Probabilites
0,CBL_000001,1,OP_05 (0.72),"OP_05: 0.72, OP_06: 0.34, OP_07: 0.16"
1,CBL_000002,0,Aucune,"OP_05: 0.38, OP_06: 0.30, OP_02: 0.16"
2,CBL_000003,1,MACH_01 (0.81),"MACH_01: 0.81, OP_02: 0.52, MACH_02: 0.41"
3,CBL_000004,0,Aucune,"MACH_01: 0.25, INSP_03: 0.24, INSP_02: 0.19"
4,CBL_000005,0,Aucune,"INSP_02: 0.27, MACH_05: 0.24, INSP_03: 0.23"
5,CBL_000006,1,MACH_01 (0.62),"MACH_01: 0.62, INSP_03: 0.19, MACH_03: 0.12"
6,CBL_000007,0,Aucune,"MACH_01: 0.17, INSP_02: 0.09, SENS_04: 0.06"
7,CBL_000008,0,Aucune,"INSP_02: 0.28, INSP_01: 0.15, MACH_01: 0.08"
8,CBL_000009,0,Aucune,"INSP_02: 0.33, MACH_04: 0.29, INSP_01: 0.10"
9,CBL_000010,0,Aucune,"MACH_01: 0.40, MACH_05: 0.25, MACH_03: 0.20"


## 3. Mode 2 : Tester une prédiction sur un scénario manuel (nouveaux événements simulés)

In [9]:
# Exemple d'un lot personnalisé composé de 3 événements successifs
custom_events = pd.DataFrame([
    {
        'Batch_ID': 'TEST_BATCH_CUSTOM_01',
        'Step_Order': 1, 'Total_Cable_Steps': 3, 'Num_Machine_Ops': 2, 'Num_Manual_Ops': 1,
        'Hours_Into_Shift': 5.5, 'Shift': 'Matin', 'Operation_Mode': 'Auto', 'Operation_Type': 'Denudage',
        'Machine_ID': 'MACH_02', 'Machine_Hours_Since_Maint': 420.0, 'Machine_Vibration_mm_s': 4.8,
        'Machine_Temp_C': 78.0, 'Machine_Cycle_Time_sec': 14.5,
        'Operator_ID': 'NONE', 'Operator_Exp_Months': 0, 'Operator_Handling_Time_sec': 0.0,
        'Sensor_ID': 'SENS_02', 'Sensor_Calibration_Drift': 0.18, 'Sensor_Anomaly_Score': 0.85, 'Sensor_Error_Flag': 1,
        'Inspector_ID': 'NONE', 'Control_Mode': 'Auto', 'Inspection_Duration_sec': 0.0,
        'Human_Error_Flag': 0, 'Step_Observed_Error': 1, 'Confirmed_Physical_Defect': 1
    },
    {
        'Batch_ID': 'TEST_BATCH_CUSTOM_01',
        'Step_Order': 2, 'Total_Cable_Steps': 3, 'Num_Machine_Ops': 2, 'Num_Manual_Ops': 1,
        'Hours_Into_Shift': 5.6, 'Shift': 'Matin', 'Operation_Mode': 'Auto', 'Operation_Type': 'Sertissage',
        'Machine_ID': 'MACH_05', 'Machine_Hours_Since_Maint': 110.0, 'Machine_Vibration_mm_s': 1.2,
        'Machine_Temp_C': 46.0, 'Machine_Cycle_Time_sec': 11.0,
        'Operator_ID': 'NONE', 'Operator_Exp_Months': 0, 'Operator_Handling_Time_sec': 0.0,
        'Sensor_ID': 'SENS_01', 'Sensor_Calibration_Drift': 0.02, 'Sensor_Anomaly_Score': 0.10, 'Sensor_Error_Flag': 0,
        'Inspector_ID': 'NONE', 'Control_Mode': 'Auto', 'Inspection_Duration_sec': 0.0,
        'Human_Error_Flag': 0, 'Step_Observed_Error': 0, 'Confirmed_Physical_Defect': 0
    },
    {
        'Batch_ID': 'TEST_BATCH_CUSTOM_01',
        'Step_Order': 3, 'Total_Cable_Steps': 3, 'Num_Machine_Ops': 2, 'Num_Manual_Ops': 1,
        'Hours_Into_Shift': 5.7, 'Shift': 'Matin', 'Operation_Mode': 'Manuel', 'Operation_Type': 'Insertion_Connecteur',
        'Machine_ID': 'NONE', 'Machine_Hours_Since_Maint': 0.0, 'Machine_Vibration_mm_s': 0.0,
        'Machine_Temp_C': 0.0, 'Machine_Cycle_Time_sec': 0.0,
        'Operator_ID': 'OP_05', 'Operator_Exp_Months': 4, 'Operator_Handling_Time_sec': 29.0,
        'Sensor_ID': 'NONE', 'Sensor_Calibration_Drift': 0.0, 'Sensor_Anomaly_Score': 0.0, 'Sensor_Error_Flag': 0,
        'Inspector_ID': 'INSP_01', 'Control_Mode': 'Manuel', 'Inspection_Duration_sec': 18.0,
        'Human_Error_Flag': 1, 'Step_Observed_Error': 1, 'Confirmed_Physical_Defect': 1
    }
])

custom_probs, custom_summary = predict_error_sources(custom_events, group_col='Batch_ID', threshold=0.55)
print('=== Résultat sur le scénario personnalisé ===')
display(custom_summary)

ranked_entities = custom_probs.iloc[0].sort_values(ascending=False).to_frame(name='Probabilite')
display(ranked_entities[ranked_entities['Probabilite'] >= 0.10].round(3))

=== Résultat sur le scénario personnalisé ===


,Batch_ID,Nb_Sources_Detectees,Sources_Detectees,Top_3_Probabilites
0,TEST_BATCH_CUSTOM_01,3,"MACH_02 (1.00), OP_01 (0.92), OP_02 (0.62)","MACH_02: 1.00, OP_01: 0.92, OP_02: 0.62"


,Probabilite
MACH_02,0.997
OP_01,0.920
OP_02,0.623
OP_07,0.405
OP_06,0.305
OP_05,0.293
MACH_01,0.291
OP_08,0.237
OP_03,0.224
SENS_02,0.163


## 4. Export des prédictions vers un fichier CSV

In [10]:
OUTPUT_CSV = 'predictions_detection_error_src.csv'
export_df = summary_batch_df.merge(probs_batch_df.round(4).reset_index(), on='Batch_ID')
export_df.to_csv(OUTPUT_CSV, index=False)
print(f'Prédictions exportées dans : {OUTPUT_CSV} ({len(export_df)} lignes)')

Prédictions exportées dans : predictions_detection_error_src.csv (15 lignes)
